## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import ttest_1samp, ttest_ind, f_oneway

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
import statsmodels.api as sm
import statsmodels.formula.api as smf

from scipy.optimize import linprog

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")


## 2. Load and Inspect Data

In [ ]:
# Update this path if your CSV is stored elsewhere.
DATA_PATH = "train_ticket_sales_india(1).csv"

df = pd.read_csv(DATA_PATH)
df["Sale_Date"] = pd.to_datetime(df["Sale_Date"])

print("First 5 rows:")
display(df.head())

print("Last 5 rows:")
display(df.tail())

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nDescriptive summary:")
display(df.describe(include="all"))


## 3. Exploratory Data Analysis

In [ ]:
# Tickets by Coach Type
df["Coach_Type"].value_counts().plot(kind="bar")
plt.title("Tickets by Coach Type")
plt.xlabel("Coach Type")
plt.ylabel("Number of Tickets")
plt.show()

# Tickets by Payment Mode
df["Payment_Mode"].value_counts().plot(kind="bar")
plt.title("Tickets by Payment Mode")
plt.xlabel("Payment Mode")
plt.ylabel("Number of Tickets")
plt.show()

# Passenger Age Distribution
plt.hist(df["Passenger_Age"], bins=15)
plt.title("Passenger Age Distribution")
plt.xlabel("Passenger Age")
plt.ylabel("Frequency")
plt.show()

# Ticket Price Distribution
plt.hist(df["Ticket_Price"], bins=20)
plt.title("Ticket Price Distribution")
plt.xlabel("Ticket Price")
plt.ylabel("Frequency")
plt.show()

# Ticket Price Box Plot
sns.boxplot(y=df["Ticket_Price"])
plt.title("Ticket Price Box Plot")
plt.show()

# Coach Type Pie Chart
df["Coach_Type"].value_counts().plot(kind="pie", autopct="%1.1f%%")
plt.title("Ticket Distribution by Coach Type")
plt.ylabel("")
plt.show()

# Age vs Ticket Price
plt.scatter(df["Passenger_Age"], df["Ticket_Price"], alpha=0.5)
plt.title("Passenger Age vs Ticket Price")
plt.xlabel("Passenger Age")
plt.ylabel("Ticket Price")
plt.show()


## 4. Descriptive Statistics and Correlation

In [ ]:
def descriptive_statistics(column):
    data = df[column]
    return pd.Series({
        "Mean": data.mean(),
        "Median": data.median(),
        "Mode": data.mode()[0],
        "Minimum": data.min(),
        "Maximum": data.max(),
        "Range": data.max() - data.min(),
        "Variance": data.var(),
        "Standard Deviation": data.std()
    })

print("Passenger Age:")
display(descriptive_statistics("Passenger_Age"))

print("Ticket Price:")
display(descriptive_statistics("Ticket_Price"))

corr = df[["Passenger_Age", "Ticket_Price"]].corr()
print("Correlation matrix:")
display(corr)

sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

print("Ticket price skewness:", df["Ticket_Price"].skew())
print("Passenger age skewness:", df["Passenger_Age"].skew())


## 5. Linear Regression

In [ ]:
X = df[["Passenger_Age"]]
y = df["Ticket_Price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)

mse = mean_squared_error(y_test, pred)
rmse = np.sqrt(mse)

print("Coefficient:", model.coef_[0])
print("Intercept:", model.intercept_)
print("R2:", r2_score(y_test, pred))
print("MSE:", mse)
print("RMSE:", rmse)

print(f"Equation: Price = {model.intercept_:.2f} + ({model.coef_[0]:.2f} × Age)")


## 6. Probability and Distributions

In [ ]:
print("Coach-type probabilities:")
print(df["Coach_Type"].value_counts(normalize=True))

p_ac = (df["Coach_Type"] == "AC").mean()
print("P(AC):", p_ac)

# Demonstration distributions
normal_data = np.random.normal(
    df["Ticket_Price"].mean(),
    df["Ticket_Price"].std(),
    1000
)

uniform_data = np.random.uniform(
    df["Ticket_Price"].min(),
    df["Ticket_Price"].max(),
    1000
)

exponential_data = np.random.exponential(
    df["Ticket_Price"].mean(),
    1000
)

for data, title in [
    (normal_data, "Normal"),
    (uniform_data, "Uniform"),
    (exponential_data, "Exponential")
]:
    plt.hist(data, bins=30)
    plt.title(f"{title} Distribution")
    plt.xlabel("Value")
    plt.ylabel("Frequency")
    plt.show()


## 7. Sampling, Confidence Interval and Bootstrap

In [ ]:
population = df["Ticket_Price"]

# Sampling distribution
sample_means = []
for i in range(100):
    sample = population.sample(50, random_state=100 + i)
    sample_means.append(sample.mean())

sample_means = np.array(sample_means)

print("Mean of sample means:", sample_means.mean())
print("SD:", sample_means.std(ddof=1))
print("SE:", sample_means.std(ddof=1) / np.sqrt(50))

plt.hist(sample_means, bins=15)
plt.title("Sampling Distribution of Mean Ticket Price")
plt.xlabel("Sample Mean")
plt.ylabel("Frequency")
plt.show()

# Confidence intervals
for confidence in [0.90, 0.95, 0.99]:
    ci = stats.t.interval(
        confidence,
        len(population) - 1,
        loc=population.mean(),
        scale=stats.sem(population)
    )
    print(f"{int(confidence * 100)}% CI:", ci)

# Bootstrap 95% CI for median
rng = np.random.default_rng(42)
boots = np.median(
    rng.choice(
        population.values,
        size=(1000, len(population)),
        replace=True
    ),
    axis=1
)

bootstrap_ci = np.percentile(boots, [2.5, 97.5])
print("Bootstrap 95% CI:", bootstrap_ci)


## 8. Hypothesis Testing and A/B-Style Testing

In [ ]:
# One-sample t-test: H0: mean ticket price = 1000
t_stat, p_value = ttest_1samp(df["Ticket_Price"], 1000)

print("One-sample t:", t_stat)
print("p-value:", p_value)

# A/B-style comparison: AC vs Sleeper ticket prices
ac = df.loc[df["Coach_Type"] == "AC", "Ticket_Price"]
sleeper = df.loc[df["Coach_Type"] == "Sleeper", "Ticket_Price"]

t_stat, p_value = ttest_ind(ac, sleeper, equal_var=False)

print("AC mean:", ac.mean())
print("Sleeper mean:", sleeper.mean())
print("Welch t:", t_stat)
print("p-value:", p_value)


## 9. Time Series, Forecasting and Accuracy

In [ ]:
daily_sales = (
    df.groupby("Sale_Date")
      .size()
      .rename("Ticket_Sales")
      .sort_index()
)

print("Number of observed dates:", len(daily_sales))

plt.plot(daily_sales)
plt.title("Daily Train Ticket Sales")
plt.xlabel("Sale Date")
plt.ylabel("Tickets Sold")
plt.show()

# Hold out the last 30 days for accuracy evaluation
train = daily_sales.iloc[:-30]
test = daily_sales.iloc[-30:]

# Exponential smoothing
es = ExponentialSmoothing(
    train,
    trend="add",
    seasonal=None
).fit()

es_pred = es.forecast(30)

mae = mean_absolute_error(test, es_pred)
mse = mean_squared_error(test, es_pred)
rmse = np.sqrt(mse)

print("Exponential Smoothing MAE:", mae)
print("Exponential Smoothing MSE:", mse)
print("Exponential Smoothing RMSE:", rmse)

# ARIMA demonstration
arima = ARIMA(daily_sales, order=(1, 1, 1)).fit()
arima_forecast = arima.forecast(30)

print("ARIMA 30-day forecast:")
display(arima_forecast)


## 10. Optimization

The source project does not contain actual profit or capacity fields, so this section uses the **hypothetical assumptions stated in the project**:

- AC profit = 800
- Sleeper profit = 400
- General profit = 150
- Total capacity = 1,000
- Maximum AC = 400
- Maximum Sleeper = 400
- Maximum General = 500


In [ ]:
profit = np.array([800, 400, 150])

result = linprog(
    -profit,
    A_ub=[[1, 1, 1]],
    b_ub=[1000],
    bounds=[(0, 400), (0, 400), (0, 500)],
    method="highs"
)

print("Optimal allocation:", result.x)
print("Maximum assumed profit:", -result.fun)


## 11. Simulation

In [ ]:
np.random.seed(42)
average_daily = daily_sales.mean()

for name, multiplier in {
    "Low Demand": 0.8,
    "Normal Demand": 1.0,
    "High Demand": 1.2
}.items():
    demand = np.random.poisson(
        average_daily * multiplier,
        1000
    )
    print(
        name,
        "| Average:", demand.mean(),
        "| Minimum:", demand.min(),
        "| Maximum:", demand.max()
    )


## 12. ANOVA and Factorial Analysis

In [ ]:
# One-way ANOVA by coach type
groups = [
    group["Ticket_Price"].values
    for _, group in df.groupby("Coach_Type")
]

f_stat, p_value = f_oneway(*groups)

print("ANOVA F:", f_stat)
print("ANOVA p-value:", p_value)

# Two-factor factorial-style ANOVA
factorial_model = smf.ols(
    "Ticket_Price ~ C(Coach_Type) * C(Payment_Mode)",
    data=df
).fit()

anova_table = sm.stats.anova_lm(factorial_model, typ=2)

print("Factorial ANOVA table:")
display(anova_table)

# Visualization
sns.boxplot(
    data=df,
    x="Coach_Type",
    y="Ticket_Price",
    hue="Payment_Mode"
)
plt.title("Ticket Price by Coach Type and Payment Mode")
plt.show()


## 13. Business Summary

In [ ]:
summary = {
    "Total tickets": len(df),
    "Total ticket amount": df["Ticket_Price"].sum(),
    "Average ticket price": df["Ticket_Price"].mean(),
    "Average passenger age": df["Passenger_Age"].mean(),
    "Most common coach": df["Coach_Type"].mode()[0],
    "Most common payment mode": df["Payment_Mode"].mode()[0],
    "AC probability": (df["Coach_Type"] == "AC").mean(),
    "Age-price correlation": df[["Passenger_Age", "Ticket_Price"]].corr().loc[
        "Passenger_Age", "Ticket_Price"
    ]
}

for key, value in summary.items():
    print(f"{key}: {value}")


## Conclusion

The notebook reproduces the analytical workflow described in the supplied project. The analysis covers preprocessing, EDA, descriptive statistics, distribution analysis, correlation, regression, probability, sampling, confidence intervals, bootstrapping, hypothesis testing, A/B-style testing, time-series forecasting, optimization, simulation, ANOVA, and factorial analysis.

The optimization section is explicitly hypothetical because the supplied dataset does not contain profit/capacity fields. Likewise, the A/B-style comparison is not a true conversion-rate experiment because visitor/exposure and conversion variables are absent.
